# 3. 多流域多年降雨量计算（串行与并行对比）

本教程将演示如何使用**串行**和**并行**方法处理大规模多流域、多年降雨数据，并对比其性能。

## 目标

1. 实现多流域批量处理功能
2. 实现多年时间序列处理
3. 对比串行与并行计算性能
4. 统计计算耗时和资源使用
5. 为大规模数据处理提供优化方案

## 数据规模

- **流域数量**: 5819 个流域
- **时间跨度**: 1年示例，可扩展至 40 年
- **网格数据**: CHIRPS 0.05° 分辨率
- **总数据量**: 估计数百 GB

---

## 1. 导入必要的库

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from shapely.geometry import box
from shapely.ops import unary_union
import warnings
import time
from datetime import datetime
import os
from pathlib import Path
import json

# 并行计算库
from multiprocessing import Pool, cpu_count, Manager
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
import dask
import dask.array as da
from dask.diagnostics import ProgressBar

warnings.filterwarnings('ignore')

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

print("所有库导入成功！")
print(f"当前时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"可用CPU核心数: {cpu_count()}")

## 2. 读取流域数据

In [ ]:
# 读取流域 shapefile
shapefile_path = "shapes/basins.shp"
basins_gdf = gpd.read_file(shapefile_path)

print(f"成功读取流域数据！")
print(f"流域总数: {len(basins_gdf)}")
print(f"坐标系统: {basins_gdf.crs}")
print(f"\n前3个流域信息:")
print(basins_gdf[['BASIN_ID']].head(3))

## 3. 创建模拟降雨网格数据

为演示目的，创建覆盖所有流域的模拟降雨网格数据

In [ ]:
def create_global_precipitation_dataset(basins_gdf, resolution=0.05, days=365):
    """
    创建覆盖所有流域的全球降雨数据集
    
    参数:
    - basins_gdf: 流域 GeoDataFrame
    - resolution: 网格分辨率（度）
    - days: 时间天数
    """
    print("创建全球降雨网格数据...")
    
    # 获取所有流域的总边界
    total_bounds = basins_gdf.total_bounds
    minx, miny, maxx, maxy = total_bounds
    
    # 扩展边界
    minx -= resolution * 2
    miny -= resolution * 2
    maxx += resolution * 2
    maxy += resolution * 2
    
    print(f"数据范围: 经度 [{minx:.2f}, {maxx:.2f}], 纬度 [{miny:.2f}, {maxy:.2f}]")
    
    # 创建网格坐标
    lons = np.arange(minx, maxx, resolution)
    lats = np.arange(miny, maxy, resolution)
    
    print(f"网格大小: {len(lats)} × {len(lons)} = {len(lats) * len(lons)} 个网格")
    
    # 创建时间序列
    times = pd.date_range('2024-01-01', periods=days, freq='D')
    
    # 使用 Dask 数组创建大规模数据（延迟计算）
    print(f"创建 {days} 天的降雨数据 (使用 Dask 延迟计算)...")
    
    # 分块大小
    chunk_time = min(30, days)
    chunk_lat = min(50, len(lats))
    chunk_lon = min(50, len(lons))
    
    # 使用 Dask 创建随机降雨数据
    precipitation_dask = da.random.gamma(
        2, 3, 
        size=(len(times), len(lats), len(lons)),
        chunks=(chunk_time, chunk_lat, chunk_lon)
    )
    
    # 创建 xarray Dataset
    ds = xr.Dataset(
        {
            'precipitation': (['time', 'lat', 'lon'], precipitation_dask)
        },
        coords={
            'time': times,
            'lat': lats,
            'lon': lons
        }
    )
    
    ds['precipitation'].attrs = {
        'units': 'mm/day',
        'long_name': 'Daily Precipitation',
        'source': 'Simulated data'
    }
    
    data_size_gb = (len(times) * len(lats) * len(lons) * 8) / (1024**3)
    print(f"数据总大小: {data_size_gb:.2f} GB (延迟加载)")
    print(f"分块信息: 时间={chunk_time}, 纬度={chunk_lat}, 经度={chunk_lon}")
    
    return ds

# 创建全球降雨数据集
start_time = time.time()
precip_ds = create_global_precipitation_dataset(basins_gdf, resolution=0.05, days=365)
creation_time = time.time() - start_time

print(f"\n数据集创建完成，耗时: {creation_time:.2f} 秒")
print(f"\n数据集概览:")
print(precip_ds)

## 4. 面积占比法核心函数

定义单个流域的面积权重计算和平均降雨计算函数

In [ ]:
def calculate_basin_weights(basin_geom, lons, lats, resolution):
    """
    计算流域的网格面积权重
    
    参数:
    - basin_geom: 流域几何对象
    - lons: 经度数组
    - lats: 纬度数组
    - resolution: 网格分辨率
    
    返回:
    - dict: 包含网格索引和权重的字典
    """
    # 获取流域边界
    minx, miny, maxx, maxy = basin_geom.bounds
    
    # 找到相关的网格索引
    lat_mask = (lats >= miny - resolution) & (lats <= maxy + resolution)
    lon_mask = (lons >= minx - resolution) & (lons <= maxx + resolution)
    
    relevant_lats = lats[lat_mask]
    relevant_lons = lons[lon_mask]
    
    lat_indices = np.where(lat_mask)[0]
    lon_indices = np.where(lon_mask)[0]
    
    # 创建网格并计算交集
    weights_data = []
    
    # 转换到投影坐标系
    basin_gdf_temp = gpd.GeoDataFrame({'geometry': [basin_geom]}, crs='EPSG:4326')
    basin_proj = basin_gdf_temp.to_crs('ESRI:102025')
    basin_geom_proj = basin_proj.geometry.values[0]
    
    for i, lat in enumerate(relevant_lats):
        for j, lon in enumerate(relevant_lons):
            # 创建网格单元
            grid_cell = box(
                lon - resolution/2,
                lat - resolution/2,
                lon + resolution/2,
                lat + resolution/2
            )
            
            # 检查是否相交
            if grid_cell.intersects(basin_geom):
                # 转换到投影坐标系计算面积
                grid_gdf = gpd.GeoDataFrame({'geometry': [grid_cell]}, crs='EPSG:4326')
                grid_proj = grid_gdf.to_crs('ESRI:102025')
                grid_geom_proj = grid_proj.geometry.values[0]
                
                try:
                    intersection = grid_geom_proj.intersection(basin_geom_proj)
                    area_km2 = intersection.area / 1e6
                    
                    if area_km2 > 0:
                        weights_data.append({
                            'lat_idx': lat_indices[i],
                            'lon_idx': lon_indices[j],
                            'area_km2': area_km2
                        })
                except:
                    continue
    
    if not weights_data:
        return None
    
    # 归一化权重
    total_area = sum(w['area_km2'] for w in weights_data)
    
    for w in weights_data:
        w['weight'] = w['area_km2'] / total_area
    
    return {
        'weights_data': weights_data,
        'total_area_km2': total_area,
        'grid_count': len(weights_data)
    }


def calculate_basin_average_from_weights(precip_data, weights_info):
    """
    根据预计算的权重计算流域平均降雨量
    
    参数:
    - precip_data: 降雨数据 (xarray.Dataset)
    - weights_info: 权重信息字典
    
    返回:
    - pandas.Series: 流域平均降雨时间序列
    """
    if weights_info is None:
        return None
    
    weights_data = weights_info['weights_data']
    n_timesteps = len(precip_data.time)
    
    # 初始化结果数组
    basin_avg = np.zeros(n_timesteps)
    
    # 提取相关网格数据并计算加权平均
    for w in weights_data:
        lat_idx = w['lat_idx']
        lon_idx = w['lon_idx']
        weight = w['weight']
        
        # 提取该网格的时间序列
        grid_precip = precip_data.precipitation[:, lat_idx, lon_idx].values
        
        # 加权累加
        basin_avg += grid_precip * weight
    
    # 转换为 pandas Series
    basin_avg_series = pd.Series(
        basin_avg,
        index=pd.to_datetime(precip_data.time.values),
        name='basin_avg_precipitation'
    )
    
    return basin_avg_series


print("核心函数定义完成！")

## 5. 串行处理实现

In [ ]:
def process_basins_serial(basins_gdf, precip_ds, basin_ids=None, save_results=True):
    """
    串行处理多个流域
    
    参数:
    - basins_gdf: 流域 GeoDataFrame
    - precip_ds: 降雨数据集
    - basin_ids: 要处理的流域 ID 列表（None 表示全部）
    - save_results: 是否保存结果
    
    返回:
    - dict: 处理结果和统计信息
    """
    print("=" * 70)
    print("串行处理开始")
    print("=" * 70)
    
    if basin_ids is None:
        basins_to_process = basins_gdf
    else:
        basins_to_process = basins_gdf[basins_gdf['BASIN_ID'].isin(basin_ids)]
    
    n_basins = len(basins_to_process)
    print(f"待处理流域数: {n_basins}")
    
    resolution = float(precip_ds.lon.values[1] - precip_ds.lon.values[0])
    lons = precip_ds.lon.values
    lats = precip_ds.lat.values
    
    results = {}
    processing_times = []
    
    overall_start = time.time()
    
    for idx, (_, basin_row) in enumerate(basins_to_process.iterrows(), 1):
        basin_id = basin_row['BASIN_ID']
        basin_geom = basin_row['geometry']
        
        basin_start = time.time()
        
        try:
            # 计算权重
            weights_info = calculate_basin_weights(basin_geom, lons, lats, resolution)
            
            if weights_info is None:
                print(f"[{idx}/{n_basins}] {basin_id}: 无相交网格，跳过")
                continue
            
            # 计算平均降雨
            basin_avg = calculate_basin_average_from_weights(precip_ds, weights_info)
            
            basin_time = time.time() - basin_start
            processing_times.append(basin_time)
            
            results[basin_id] = {
                'basin_avg_precip': basin_avg,
                'total_area_km2': weights_info['total_area_km2'],
                'grid_count': weights_info['grid_count'],
                'processing_time': basin_time,
                'annual_total_mm': basin_avg.sum(),
                'daily_mean_mm': basin_avg.mean()
            }
            
            if idx % 10 == 0 or idx == n_basins:
                elapsed = time.time() - overall_start
                avg_time = np.mean(processing_times)
                eta = avg_time * (n_basins - idx)
                
                print(f"[{idx}/{n_basins}] {basin_id}: "
                      f"耗时 {basin_time:.2f}s, "
                      f"网格数 {weights_info['grid_count']}, "
                      f"年降雨 {basin_avg.sum():.1f}mm | "
                      f"已用时 {elapsed:.1f}s, 预计剩余 {eta:.1f}s")
        
        except Exception as e:
            print(f"[{idx}/{n_basins}] {basin_id}: 处理失败 - {e}")
            continue
    
    total_time = time.time() - overall_start
    
    # 统计信息
    stats = {
        'n_basins_total': n_basins,
        'n_basins_processed': len(results),
        'total_time_seconds': total_time,
        'avg_time_per_basin': np.mean(processing_times) if processing_times else 0,
        'min_time_per_basin': np.min(processing_times) if processing_times else 0,
        'max_time_per_basin': np.max(processing_times) if processing_times else 0,
        'processing_speed': len(results) / total_time if total_time > 0 else 0
    }
    
    print("\n" + "=" * 70)
    print("串行处理完成")
    print("=" * 70)
    print(f"处理流域数: {stats['n_basins_processed']} / {stats['n_basins_total']}")
    print(f"总耗时: {stats['total_time_seconds']:.2f} 秒 ({stats['total_time_seconds']/60:.2f} 分钟)")
    print(f"平均每流域耗时: {stats['avg_time_per_basin']:.3f} 秒")
    print(f"处理速度: {stats['processing_speed']:.2f} 流域/秒")
    print("=" * 70 + "\n")
    
    return {'results': results, 'stats': stats}


print("串行处理函数定义完成！")

## 6. 并行处理实现

In [ ]:
def process_single_basin_worker(args):
    """
    单个流域处理的工作函数（用于并行）
    
    参数:
    - args: (basin_id, basin_geom, lons, lats, resolution, precip_data_subset)
    
    返回:
    - tuple: (basin_id, result_dict) 或 (basin_id, None)
    """
    basin_id, basin_geom, lons, lats, resolution, precip_data = args
    
    start_time = time.time()
    
    try:
        # 计算权重
        weights_info = calculate_basin_weights(basin_geom, lons, lats, resolution)
        
        if weights_info is None:
            return (basin_id, None)
        
        # 计算平均降雨
        basin_avg = calculate_basin_average_from_weights(precip_data, weights_info)
        
        processing_time = time.time() - start_time
        
        result = {
            'basin_avg_precip': basin_avg,
            'total_area_km2': weights_info['total_area_km2'],
            'grid_count': weights_info['grid_count'],
            'processing_time': processing_time,
            'annual_total_mm': basin_avg.sum(),
            'daily_mean_mm': basin_avg.mean()
        }
        
        return (basin_id, result)
    
    except Exception as e:
        return (basin_id, None)


def process_basins_parallel(basins_gdf, precip_ds, basin_ids=None, n_workers=None):
    """
    并行处理多个流域
    
    参数:
    - basins_gdf: 流域 GeoDataFrame
    - precip_ds: 降雨数据集
    - basin_ids: 要处理的流域 ID 列表（None 表示全部）
    - n_workers: 并行工作进程数（None 表示使用 CPU 核心数）
    
    返回:
    - dict: 处理结果和统计信息
    """
    if n_workers is None:
        n_workers = cpu_count()
    
    print("=" * 70)
    print("并行处理开始")
    print("=" * 70)
    
    if basin_ids is None:
        basins_to_process = basins_gdf
    else:
        basins_to_process = basins_gdf[basins_gdf['BASIN_ID'].isin(basin_ids)]
    
    n_basins = len(basins_to_process)
    print(f"待处理流域数: {n_basins}")
    print(f"并行工作进程数: {n_workers}")
    
    resolution = float(precip_ds.lon.values[1] - precip_ds.lon.values[0])
    lons = precip_ds.lon.values
    lats = precip_ds.lat.values
    
    # 准备任务列表
    tasks = []
    for _, basin_row in basins_to_process.iterrows():
        basin_id = basin_row['BASIN_ID']
        basin_geom = basin_row['geometry']
        
        tasks.append((
            basin_id,
            basin_geom,
            lons,
            lats,
            resolution,
            precip_ds
        ))
    
    results = {}
    processing_times = []
    
    overall_start = time.time()
    
    # 使用 ProcessPoolExecutor 并行处理
    with ProcessPoolExecutor(max_workers=n_workers) as executor:
        # 提交所有任务
        future_to_basin = {executor.submit(process_single_basin_worker, task): task[0] 
                          for task in tasks}
        
        # 收集结果
        completed = 0
        for future in as_completed(future_to_basin):
            basin_id = future_to_basin[future]
            completed += 1
            
            try:
                basin_id_result, result = future.result()
                
                if result is not None:
                    results[basin_id_result] = result
                    processing_times.append(result['processing_time'])
                    
                    if completed % 10 == 0 or completed == n_basins:
                        elapsed = time.time() - overall_start
                        progress = completed / n_basins * 100
                        eta = elapsed / completed * (n_basins - completed) if completed > 0 else 0
                        
                        print(f"进度: {completed}/{n_basins} ({progress:.1f}%) | "
                              f"已用时 {elapsed:.1f}s, 预计剩余 {eta:.1f}s | "
                              f"成功 {len(results)} 个")
                
            except Exception as e:
                print(f"流域 {basin_id} 处理失败: {e}")
    
    total_time = time.time() - overall_start
    
    # 统计信息
    stats = {
        'n_basins_total': n_basins,
        'n_basins_processed': len(results),
        'total_time_seconds': total_time,
        'avg_time_per_basin': np.mean(processing_times) if processing_times else 0,
        'min_time_per_basin': np.min(processing_times) if processing_times else 0,
        'max_time_per_basin': np.max(processing_times) if processing_times else 0,
        'processing_speed': len(results) / total_time if total_time > 0 else 0,
        'n_workers': n_workers
    }
    
    print("\n" + "=" * 70)
    print("并行处理完成")
    print("=" * 70)
    print(f"处理流域数: {stats['n_basins_processed']} / {stats['n_basins_total']}")
    print(f"总耗时: {stats['total_time_seconds']:.2f} 秒 ({stats['total_time_seconds']/60:.2f} 分钟)")
    print(f"平均每流域耗时: {stats['avg_time_per_basin']:.3f} 秒")
    print(f"处理速度: {stats['processing_speed']:.2f} 流域/秒")
    print(f"并行效率: {stats['processing_speed'] / n_workers:.2f} 流域/秒/核心")
    print("=" * 70 + "\n")
    
    return {'results': results, 'stats': stats}


print("并行处理函数定义完成！")

## 7. 小规模测试（10个流域）

In [ ]:
# 选择前10个流域进行测试
test_basin_ids = basins_gdf['BASIN_ID'].head(10).tolist()

print(f"测试流域数: {len(test_basin_ids)}")
print(f"测试流域 ID: {test_basin_ids[:5]}...\n")

# 串行处理测试
serial_result = process_basins_serial(basins_gdf, precip_ds, basin_ids=test_basin_ids, save_results=False)

# 并行处理测试
parallel_result = process_basins_parallel(basins_gdf, precip_ds, basin_ids=test_basin_ids, n_workers=4)

## 8. 性能对比分析

In [ ]:
def compare_performance(serial_stats, parallel_stats):
    """
    对比串行和并行处理性能
    """
    print("=" * 70)
    print("性能对比分析")
    print("=" * 70)
    
    serial_time = serial_stats['total_time_seconds']
    parallel_time = parallel_stats['total_time_seconds']
    speedup = serial_time / parallel_time if parallel_time > 0 else 0
    
    n_workers = parallel_stats.get('n_workers', 1)
    efficiency = (speedup / n_workers) * 100 if n_workers > 0 else 0
    
    print(f"\n【处理模式对比】")
    print(f"  串行处理:")
    print(f"    总耗时: {serial_time:.2f} 秒 ({serial_time/60:.2f} 分钟)")
    print(f"    处理速度: {serial_stats['processing_speed']:.2f} 流域/秒")
    print(f"    平均每流域: {serial_stats['avg_time_per_basin']:.3f} 秒")
    
    print(f"\n  并行处理 ({n_workers} 核心):")
    print(f"    总耗时: {parallel_time:.2f} 秒 ({parallel_time/60:.2f} 分钟)")
    print(f"    处理速度: {parallel_stats['processing_speed']:.2f} 流域/秒")
    print(f"    平均每流域: {parallel_stats['avg_time_per_basin']:.3f} 秒")
    
    print(f"\n【性能提升】")
    print(f"  加速比: {speedup:.2f}x")
    print(f"  时间节省: {serial_time - parallel_time:.2f} 秒 ({(1 - parallel_time/serial_time)*100:.1f}%)")
    print(f"  并行效率: {efficiency:.1f}%")
    
    # 可视化对比
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # 1. 总耗时对比
    ax1 = axes[0]
    methods = ['串行', f'并行\n({n_workers}核心)']
    times = [serial_time, parallel_time]
    colors = ['steelblue', 'coral']
    bars = ax1.bar(methods, times, color=colors, edgecolor='black', alpha=0.7)
    ax1.set_ylabel('耗时 (秒)', fontsize=12)
    ax1.set_title('总耗时对比', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3, axis='y')
    
    for bar, time_val in zip(bars, times):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{time_val:.2f}s',
                ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    # 2. 处理速度对比
    ax2 = axes[1]
    speeds = [serial_stats['processing_speed'], parallel_stats['processing_speed']]
    bars = ax2.bar(methods, speeds, color=colors, edgecolor='black', alpha=0.7)
    ax2.set_ylabel('处理速度 (流域/秒)', fontsize=12)
    ax2.set_title('处理速度对比', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='y')
    
    for bar, speed in zip(bars, speeds):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{speed:.2f}',
                ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    # 3. 加速比和效率
    ax3 = axes[2]
    metrics = ['加速比', '并行效率(%)']
    values = [speedup, efficiency]
    bars = ax3.bar(metrics, values, color=['green', 'purple'], edgecolor='black', alpha=0.7)
    ax3.set_title('性能指标', fontsize=14, fontweight='bold')
    ax3.grid(True, alpha=0.3, axis='y')
    
    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.2f}',
                ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print("\n" + "=" * 70)
    
    return {
        'speedup': speedup,
        'efficiency': efficiency,
        'time_saved': serial_time - parallel_time
    }

# 执行性能对比
performance_comparison = compare_performance(serial_result['stats'], parallel_result['stats'])

## 9. 大规模处理估算

In [ ]:
def estimate_large_scale_processing(stats, n_total_basins=5819, n_years=40):
    """
    估算大规模处理所需时间
    """
    print("=" * 70)
    print("大规模处理时间估算")
    print("=" * 70)
    
    avg_time_per_basin = stats['avg_time_per_basin']
    processing_speed = stats['processing_speed']
    
    # 单年处理时间估算
    est_time_1yr_seconds = n_total_basins / processing_speed
    est_time_1yr_hours = est_time_1yr_seconds / 3600
    est_time_1yr_days = est_time_1yr_hours / 24
    
    # 多年处理时间估算
    est_time_multi_yr_seconds = est_time_1yr_seconds * n_years
    est_time_multi_yr_hours = est_time_multi_yr_seconds / 3600
    est_time_multi_yr_days = est_time_multi_yr_hours / 24
    
    print(f"\n【单年数据处理估算】")
    print(f"  流域数: {n_total_basins}")
    print(f"  处理速度: {processing_speed:.2f} 流域/秒")
    print(f"  估算总耗时: {est_time_1yr_seconds:.0f} 秒")
    print(f"             = {est_time_1yr_hours:.2f} 小时")
    print(f"             = {est_time_1yr_days:.2f} 天")
    
    print(f"\n【{n_years}年数据处理估算】")
    print(f"  流域数: {n_total_basins}")
    print(f"  年数: {n_years}")
    print(f"  估算总耗时: {est_time_multi_yr_seconds:.0f} 秒")
    print(f"             = {est_time_multi_yr_hours:.2f} 小时")
    print(f"             = {est_time_multi_yr_days:.2f} 天")
    
    # 评分目标分析
    print(f"\n【课程评分目标分析】")
    
    targets = {
        '60分（及格）': {'basins_pct': 1/5819, 'years': 1},
        '70-79分（C档）': {'basins_pct': 0.55, 'years': 10},
        '80-89分（B档）': {'basins_pct': 0.85, 'years': 20},
        '90-99分（A档）': {'basins_pct': 1.0, 'years': 40},
        '100分（满分）': {'basins_pct': 1.0, 'years': 40, 'deadline_days': 7}
    }
    
    for target, params in targets.items():
        n_basins = int(n_total_basins * params['basins_pct'])
        years = params['years']
        
        est_time_sec = (n_basins / processing_speed) * years
        est_time_days = est_time_sec / 3600 / 24
        
        print(f"\n  {target}:")
        print(f"    需处理: {n_basins} 流域 × {years} 年")
        print(f"    估算耗时: {est_time_days:.2f} 天")
        
        if 'deadline_days' in params:
            deadline = params['deadline_days']
            if est_time_days <= deadline:
                print(f"    ✅ 可在 {deadline} 天内完成")
            else:
                required_speedup = est_time_days / deadline
                print(f"    ❌ 需要 {required_speedup:.1f}x 加速才能在 {deadline} 天内完成")
    
    print("\n" + "=" * 70)
    
    return {
        'est_time_1yr_days': est_time_1yr_days,
        'est_time_40yr_days': est_time_multi_yr_days
    }

# 串行处理估算
print("\n【串行处理估算】")
serial_estimates = estimate_large_scale_processing(serial_result['stats'])

# 并行处理估算
print("\n【并行处理估算】")
parallel_estimates = estimate_large_scale_processing(parallel_result['stats'])

## 10. 总结与统计

In [ ]:
print("=" * 70)
print("多流域多年降雨量计算 - 总结报告")
print("=" * 70)

print("\n【测试配置】")
print(f"  测试流域数: {len(test_basin_ids)}")
print(f"  时间跨度: 2024年 (365天)")
print(f"  网格分辨率: 0.05° (约 5.5 km)")
print(f"  CPU核心数: {cpu_count()}")
print(f"  并行工作进程: {parallel_result['stats'].get('n_workers', 4)}")

print("\n【串行处理结果】")
print(f"  成功处理: {serial_result['stats']['n_basins_processed']} 个流域")
print(f"  总耗时: {serial_result['stats']['total_time_seconds']:.2f} 秒")
print(f"  处理速度: {serial_result['stats']['processing_speed']:.2f} 流域/秒")
print(f"  平均每流域: {serial_result['stats']['avg_time_per_basin']:.3f} 秒")

print("\n【并行处理结果】")
print(f"  成功处理: {parallel_result['stats']['n_basins_processed']} 个流域")
print(f"  总耗时: {parallel_result['stats']['total_time_seconds']:.2f} 秒")
print(f"  处理速度: {parallel_result['stats']['processing_speed']:.2f} 流域/秒")
print(f"  平均每流域: {parallel_result['stats']['avg_time_per_basin']:.3f} 秒")
print(f"  加速比: {performance_comparison['speedup']:.2f}x")
print(f"  并行效率: {performance_comparison['efficiency']:.1f}%")

print("\n【大规模处理估算 - 并行模式】")
print(f"  5819个流域 × 1年:")
print(f"    估算耗时: {parallel_estimates['est_time_1yr_days']:.2f} 天")
print(f"  5819个流域 × 40年:")
print(f"    估算耗时: {parallel_estimates['est_time_40yr_days']:.2f} 天")

print("\n【优化建议】")
print("  1. 使用更多CPU核心进行并行计算")
print("  2. 预先计算并缓存流域权重矩阵")
print("  3. 使用GPU加速网格数据处理")
print("  4. 采用分布式计算框架（如Dask Distributed）")
print("  5. 优化数据I/O，使用内存映射文件")

print("\n" + "=" * 70)

# 保存统计结果
summary = {
    'test_config': {
        'n_test_basins': len(test_basin_ids),
        'time_span': '2024 (365 days)',
        'resolution': '0.05 degrees',
        'cpu_cores': cpu_count()
    },
    'serial_stats': serial_result['stats'],
    'parallel_stats': parallel_result['stats'],
    'performance_comparison': performance_comparison,
    'large_scale_estimates': {
        'serial': serial_estimates,
        'parallel': parallel_estimates
    },
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
}

# 保存为JSON
output_dir = Path('results')
output_dir.mkdir(exist_ok=True)
summary_file = output_dir / 'processing_summary.json'

# 序列化时转换numpy类型为Python原生类型
def convert_to_serializable(obj):
    if isinstance(obj, (np.int_, np.intc, np.intp, np.int8,
        np.int16, np.int32, np.int64, np.uint8,
        np.uint16, np.uint32, np.uint64)):
        return int(obj)
    elif isinstance(obj, (np.float_, np.float16, np.float32, np.float64)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

with open(summary_file, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False, default=convert_to_serializable)

print(f"\n统计结果已保存至: {summary_file}")

## 11. 小结

### 完成内容

1. ✅ **串行处理实现**: 完整的单线程顺序处理流程
2. ✅ **并行处理实现**: 基于 multiprocessing 的多核并行计算
3. ✅ **性能对比**: 详细的串行vs并行性能分析
4. ✅ **大规模估算**: 基于实测数据的全流域处理时间预测
5. ✅ **可视化分析**: 多维度性能指标可视化
6. ✅ **统计报告**: 完整的处理统计和JSON格式导出

### 关键发现

**并行处理优势**:
- **加速比**: 理想情况下可达到核心数的 60-80%
- **时间节省**: 大规模处理可节省数天甚至数周时间
- **可扩展性**: 核心数越多，提升越明显

**性能瓶颈**:
1. **I/O限制**: 数据读写可能成为瓶颈
2. **进程间通信**: 数据传输开销
3. **内存限制**: 大数据集需要careful内存管理

### 实际应用建议

**小规模数据** (< 100流域, < 5年):
- 使用串行处理即可
- 简单、易调试、资源占用低

**中规模数据** (100-1000流域, 5-20年):
- 推荐使用并行处理
- 根据CPU核心数调整工作进程
- 注意内存使用

**大规模数据** (> 1000流域, > 20年):
- 必须使用并行+分布式方案
- 考虑使用 Dask Distributed
- 分批处理，保存中间结果
- 预计算权重矩阵

### 评分目标实现路径

| 目标分数 | 数据量 | 推荐方案 | 预估时间 |
|---------|--------|---------|----------|
| 60分 | 1流域×1年 | 串行 | < 1分钟 |
| 70-79分 | 55%流域×10年 | 并行(4核) | 数小时-1天 |
| 80-89分 | 85%流域×20年 | 并行(8核) | 1-3天 |
| 90-99分 | 全部×40年 | 并行(16核) | 3-7天 |
| 100分 | 全部×40年(1周内) | 分布式(32+核) | < 7天 |

### 后续优化方向

1. **权重缓存系统**: 预计算并存储流域权重，避免重复计算
2. **增量处理**: 支持断点续传和增量更新
3. **分布式计算**: 使用 Dask Distributed 或 Spark
4. **GPU加速**: 使用 CuPy 或 RAPIDS 加速数组运算
5. **内存优化**: 使用内存映射和流式处理

---

**本教程到此结束！**

你已经掌握了多流域多年降雨量计算的串行和并行方法，可以根据实际需求选择合适的处理方案。